*[Cell 1]*

# Chapter 5: Maintaining your Delta Lake
> The following exercises use the New York Times [Covid-19 NYT Dataset](https://github.com/delta-io/delta-docs/tree/main/static/quickstart_docker/rs/data/COVID-19_NYT).

The dataset can be found in the `delta_quickstart` docker.

In [6]:
# [Cell 2]
# Databricks Free Edition: Spark runs on serverless via Databricks Connect (see dldg.py at the repo root)
from dldg import spark, DATASETS, WORK, set_conf, tree, rm, upload_dir
CH = f"{WORK}/ch05"  # replaces the Docker image's /opt/spark/work-dir
# COVID-19_NYT is a Delta table. Databricks refuses to read a Delta directory as plain parquet
# (DELTA_INVALID_FORMAT), so the reads below use format("delta") on the same files.

In [7]:
# [Cell 3]
from pyspark.sql.types import DateType
from pyspark.sql.functions import col, desc, to_date
from delta.tables import DeltaTable

In [8]:
# [Cell 4]
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.dldg.covid_nyt (
  date DATE
) USING DELTA
TBLPROPERTIES('delta.logRetentionDuration'='interval 7 days');
""")


DataFrame[]

In [9]:
# [Cell 5]
spark.sql("show tables").show()

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|    dldg|covid_nyt|      false|
+--------+---------+-----------+



In [10]:
# [Cell 6]
# will be empty on the first run. this is expected
len(spark.table("workspace.dldg.covid_nyt").inputFiles())

0

In [ ]:
# [Cell 7]
# uncomment if you'd like to begin again
#spark.sql("drop table workspace.dldg.covid_nyt")

*[Cell 8]*

## Start Populating the Table
> The next three commands are used to show Schema Evolution and Validation with Delta Lake

In [11]:
# [Cell 9]
# Populate the Table reading the Parquet covid_nyc Data
# note: this will fail on the first run, and that is okay
(spark.read
      .format("delta")
      .load(f"{DATASETS}/COVID-19_NYT")
      .withColumn("date", to_date("date", "yyyy-MM-dd"))
      .write
      .format("delta")
      .saveAsTable("workspace.dldg.covid_nyt"))

AnalysisException: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `dldg`.`covid_nyt` because it already exists.
Choose a different name, drop the existing object, add the IF NOT EXISTS clause to tolerate pre-existing objects, add the OR REPLACE clause to replace the existing materialized view, or add the OR REFRESH clause to refresh the existing streaming table. SQLSTATE: 42P07

JVM stacktrace:
org.apache.spark.sql.catalyst.analysis.TableAlreadyExistsException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.tableAlreadyExistsError(QueryCompilationErrors.scala:3379)
	at org.apache.spark.sql.execution.datasources.v2.AtomicCreateTableAsSelectExec.run(WriteToDataSourceV2Exec.scala:225)
	at org.apache.spark.sql.execution.datasources.v2.V2CommandExec.$anonfun$result$3(V2CommandExec.scala:55)
	at org.apache.spark.sql.execution.SparkPlan.runCommandInAetherOrSpark(SparkPlan.scala:231)
	at org.apache.spark.sql.execution.datasources.v2.V2CommandExec.$anonfun$result$2(V2Command

In [12]:
# [Cell 10]
# one step closer, there is still something missing...
# and yes, this operation still fails... if only...
(spark.read
      .format("delta")
      .load(f"{DATASETS}/COVID-19_NYT")
      .withColumn("date", to_date("date", "yyyy-MM-dd"))
      .write
      .format("delta")
      .mode("append")
      .saveAsTable("workspace.dldg.covid_nyt"))

AnalysisException: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: 90fb9283-5141-4526-bc1b-c7819c944d68).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- date: date (nullable = true)


Data schema:
root
 |-- date: date (nullable = true)
 |-- county: string (nullable = true)
 |-- state: string (nullable = true)
 |-- fips: integer (nullable = true)
 |-- cases: integer (nullable = true)
 |-- deaths: integer (nullable = true)


- Table ACLs are enabled in this cluster, so automatic schema migration is not allowed. Please use the ALTER TABLE command for changing the schema.

JVM stacktrace:
com.databricks.sql.transac

*[Cell 11]*

## Schema Evolution: Handle Automatically
If you trust the upstream data source (provider) then you can add the `option("mergeSchema", "true")`. Otherwise, it is better to specifically select a subset of the columns you expected to see. In this example use case, the only known column is `date`, so it is fairly safe to power ahead.

In [13]:
# [Cell 12]
# Evolve the Schema. (Showcases how to auto-merge changes to the schema)
# note: if you can trust the upstream, then this option is perfectly fine
# however, if you don't trust the upstream, then it is good to opt-in to the 
# changing columns.

(spark.read
      .format("delta")
      .load(f"{DATASETS}/COVID-19_NYT")
      .withColumn("date", to_date("date", "yyyy-MM-dd"))
      .write
      .format("delta")
      .mode("append")
      .option("mergeSchema", "true")
      .saveAsTable("workspace.dldg.covid_nyt")
    )

In [14]:
# [Cell 13]
df = spark.table("workspace.dldg.covid_nyt")
df.count()

1111930

*[Cell 14]*

# Alternatives to Auto Schema Evolution
In the previous case, we used `.option("mergeSchema", "true")` to modify the behavior of the Delta Lake writer. While this option simplifies how we evolve our Delta Lake table schemas, it comes at the price of not being fully aware of the changes to our table schema. In the case where there are unknown columns being introduced from an upstream source, you'll want to know which columns are intended to bring forward, and which columns can be safely ignored.

## Intentionally Adding Columns with Alter Table

In [15]:
# [Cell 15]
# manually set the columns. This is an example of intentional opt-in to the new columns outside of '.option("mergeSchema", "true")`. 
# Note: ADD columns fails if a column already exists (FIELD_ALREADY_EXISTS). The mergeSchema write above
# already added them, so only add the ones still missing; on a fresh table this runs the full ALTER TABLE.
new_columns = {"county": "STRING", "state": "STRING", "fips": "INT", "cases": "INT", "deaths": "INT"}
missing = [f"{name} {dtype}" for name, dtype in new_columns.items()
           if name not in spark.table("workspace.dldg.covid_nyt").columns]
if missing:
    spark.sql(f"""
    ALTER TABLE workspace.dldg.covid_nyt 
    ADD columns (
      {", ".join(missing)}
    );
    """)
else:
    print("Columns already exist (added by mergeSchema above); skipping ALTER TABLE")
# notice how we are only using `.mode("append")` and explicitly add `.option("mergeSchema", "false")`. 
# this is how we stop unwanted columns from being freely added to our Delta Lake tables. It comes at the cost of raising exceptions and failing the job.
# a failed job might seem like a bad option, but it is the cheaper option since you are intentionally blocking unknown data from flowing into your tables. 
(spark.read
      .format("delta")
      .load(f"{DATASETS}/COVID-19_NYT")
      .withColumn("date", to_date("date", "yyyy-MM-dd"))
      .write
      .format("delta")
      .option("mergeSchema", "false")
      .mode("append")
      .saveAsTable("workspace.dldg.covid_nyt"))

Columns already exist (added by mergeSchema above); skipping ALTER TABLE


In [16]:
# [Cell 16]
spark.sql("describe extended workspace.dldg.covid_nyt").show(truncate=False)

+----------------------------+----------------------------------------+-------+
|col_name                    |data_type                               |comment|
+----------------------------+----------------------------------------+-------+
|date                        |date                                    |NULL   |
|county                      |string                                  |NULL   |
|state                       |string                                  |NULL   |
|fips                        |int                                     |NULL   |
|cases                       |int                                     |NULL   |
|deaths                      |int                                     |NULL   |
|                            |                                        |       |
|# Delta Statistics Columns  |                                        |       |
|Column Names                |state, deaths, fips, county, date, cases|       |
|Column Selection Method     |first-32  

In [17]:
# [Cell 17]
spark.sql("select * from workspace.dldg.covid_nyt limit 10").show(truncate=True)

+----------+----------------+---------+-----+-----+------+
|      date|          county|    state| fips|cases|deaths|
+----------+----------------+---------+-----+-----+------+
|2021-02-25|           Caddo|Louisiana|22017|24637|   672|
|2021-02-25|       Calcasieu|Louisiana|22019|19181|   354|
|2021-02-25|        Caldwell|Louisiana|22021| 1076|    25|
|2021-02-25|         Cameron|Louisiana|22023|  543|     5|
|2021-02-25|       Catahoula|Louisiana|22025| 1053|    34|
|2021-02-25|       Claiborne|Louisiana|22027| 1407|    49|
|2021-02-25|       Concordia|Louisiana|22029| 1770|    55|
|2021-02-25|         De Soto|Louisiana|22031| 2635|    71|
|2021-02-25|East Baton Rouge|Louisiana|22033|35389|   732|
|2021-02-25|    East Carroll|Louisiana|22035| 1094|    23|
+----------+----------------+---------+-----+-----+------+



*[Cell 18]*

# Adding and Modifying Table Properties

In [18]:
# [Cell 19]
spark.sql("""
  ALTER TABLE workspace.dldg.covid_nyt 
  SET TBLPROPERTIES (
    'catalog.team_name'='dldg_authors',
    'catalog.engineering.comms.slack'='https://delta-users.slack.com/archives/CG9LR6LN4',
    'catalog.engineering.comms.email'='dldg_authors@gmail.com',
    'catalog.table.classification'='all-access'
  )""")


DataFrame[]

In [19]:
# [Cell 20]
# view the table history
from delta.tables import DeltaTable
dt = DeltaTable.forName(spark, 'workspace.dldg.covid_nyt')
dt.history(10).select("version", "timestamp", "operation").show()

+-------+-------------------+-----------------+
|version|          timestamp|        operation|
+-------+-------------------+-----------------+
|      3|2026-10-24 14:20:49|SET TBLPROPERTIES|
|      2|2026-10-24 14:13:42|            WRITE|
|      1|2026-10-24 14:08:10|            WRITE|
|      0|2026-10-24 13:38:54|     CREATE TABLE|
+-------+-------------------+-----------------+



In [21]:
# [Cell 21]
# use DeltaTable to view
dt.detail().select("properties").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|properties                                                                                                                                                                                                    

In [ ]:
# [Cell 22]
# view the table properties
spark.sql("show tblproperties workspace.dldg.covid_nyt").show(truncate=False)

*[Cell 23]*

## Removing Table Properties

In [24]:
# [Cell 24]
# add incorrect table property
# which is blocked by default
set_conf("spark.databricks.delta.allowArbitraryProperties.enabled","true")
# now we can make a mistake
# (on serverless the conf above is skipped, so the safeguard stays on and this ALTER fails as intended)
spark.sql("""
  ALTER TABLE workspace.dldg.covid_nyt
  SET TBLPROPERTIES (
    'delta.loRgetentionDuratio'='interval 7 days'
  )""")

Skipped spark.databricks.delta.allowArbitraryProperties.enabled=true: not settable on serverless compute


AnalysisException: [DELTA_UNKNOWN_CONFIGURATION] Unknown configuration was specified: delta.loRgetentionDuratio.
To disable this check, set spark.databricks.delta.allowArbitraryProperties.enabled=true in the Spark session configuration.

JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaAnalysisException
	at com.databricks.sql.transaction.tahoe.DeltaErrorsBase.unknownConfigurationKeyException(DeltaErrors.scala:1207)
	at com.databricks.sql.transaction.tahoe.DeltaErrorsBase.unknownConfigurationKeyException$(DeltaErrors.scala:1206)
	at com.databricks.sql.transaction.tahoe.DeltaErrors$.unknownConfigurationKeyException(DeltaErrors.scala:4854)
	at com.databricks.sql.transaction.tahoe.DeltaConfigsBase.validateConfiguration(DeltaConfig.scala:407)
	at com.databricks.sql.transaction.tahoe.DeltaConfigsBase.validateConfiguration$(DeltaConfig.scala:365)
	at com.databricks.sql.transaction.tahoe.DeltaConfigs$.validateConfiguration(DeltaConfig.scala:1309)
	at com.databricks.sql.transaction.tah

In [ ]:
# [Cell 25]
# luckily, we can remove the unwanted table property using UNSET
spark.sql("""
  ALTER TABLE workspace.dldg.covid_nyt 
  UNSET TBLPROPERTIES ('delta.loRgetentionDuratio')
""")
# now that we are done, let's just add back the safe guard again
set_conf("spark.databricks.delta.allowArbitraryProperties.enabled","false")

*[Cell 26]*

## Delta Table Optimization

In [ ]:
# [Cell 27]
## Creating the Small File Problem

from delta.tables import DeltaTable
(DeltaTable.createIfNotExists(spark)
    .tableName("workspace.dldg.nonoptimal_covid_nyt")
    .property("description", "table to be optimized")
    .property("catalog.team_name", "dldg_authors")
    .property("catalog.engineering.comms.slack",
	"https://delta-users.slack.com/archives/CG9LR6LN4")
    .property("catalog.engineering.comms.email","dldg_authors@gmail.com")
    .property("catalog.table.classification","all-access")
    .addColumn("date", "DATE")
    .addColumn("county", "STRING")
    .addColumn("state", "STRING")
    .addColumn("fips", "INT")
    .addColumn("cases", "INT")
    .addColumn("deaths", "INT")
    .execute())

In [ ]:
# [Cell 28]
#spark.sql("drop table workspace.dldg.nonoptimal_covid_nyt")

In [ ]:
# [Cell 29]
# you can remove `repartition(9000)` and add write...option('maxRecordsPerFile`, 10000)
# to generate more files using the DataFrameWriter
(spark
   .table("workspace.dldg.covid_nyt")
   .repartition(9000)
   .write
   .format("delta")
   .mode("overwrite")
   #.option("maxRecordsPerFile", 1000)
   .saveAsTable("workspace.dldg.nonoptimal_covid_nyt")
)

*[Cell 30]*

## Using Optimize to Fix the Small Files Problem

In [ ]:
# [Cell 31]
# set the maxFileSize to a bin-size for optimize
set_conf("spark.databricks.delta.optimize.maxFileSize", 1024*1024*1024)
(
    DeltaTable.forName(spark, "workspace.dldg.nonoptimal_covid_nyt")
    .optimize()
    .executeCompaction()
)

In [ ]:
# [Cell 32]
# Viewing the results of Optimize
from pyspark.sql.functions import col
(
    DeltaTable.forName(spark, "workspace.dldg.nonoptimal_covid_nyt")
    .history(10)
    .where(col("operation") == "OPTIMIZE")
    .select("version", "timestamp", "operation", "operationMetrics.numRemovedFiles", "operationMetrics.numAddedFiles")
    .show(truncate=False)
)


In [ ]:
# [Cell 33]
# rewind and try again
# note: the table version of the OPTIMIZE operation needs to be referenced to take the prior version
#(DeltaTable.forName(spark, "workspace.dldg.nonoptimal_covid_nyt").restoreToVersion(1))

*[Cell 34]*

## Partitioning, Repartitioning, and Default Partitions

In [ ]:
# [Cell 35]
from delta.tables import DeltaTable
from pyspark.sql.types import DateType
(DeltaTable.createIfNotExists(spark)
    .tableName("workspace.dldg.covid_nyt_by_date")
    .property("description", "table with default partitions")
    .property("catalog.team_name", "dldg_authors")
    .property("catalog.engineering.comms.slack",
	"https://delta-users.slack.com/archives/CG9LR6LN4")
    .property("catalog.engineering.comms.email","dldg_authors@gmail.com")
    .property("catalog.table.classification","all-access")
    .addColumn("date", DateType(), nullable=False)
    .addColumn("county", "STRING")
    .addColumn("state", "STRING")
    .addColumn("fips", "INT")
    .addColumn("cases", "INT")
    .addColumn("deaths", "INT")
    .partitionedBy("date")
    .execute())

In [ ]:
# [Cell 36]
# spark.sql("drop table workspace.dldg.covid_nyt_by_date")

In [ ]:
# [Cell 37]
# Use our non-partitioned source table to populate our partitioned table (automatically)
(
    spark
    .table("workspace.dldg.covid_nyt")
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "false")
    .saveAsTable("workspace.dldg.covid_nyt_by_date")
)

*[Cell 38]*

## Viewing the Partition Metadata of our Tables

In [ ]:
# [Cell 39]
spark.sql("describe extended workspace.dldg.covid_nyt_by_date").show()

In [ ]:
# [Cell 40]
# view the table metadata as a json blob

DeltaTable.forName(spark, "workspace.dldg.covid_nyt_by_date").detail().toJSON().collect()[0]

*[Cell 41]*

# Create Bronze and Silver Tables

In [ ]:
# [Cell 42]
spark.sql("show databases;").show()

In [ ]:
# [Cell 43]
# The book creates separate bronze and silver schemas here. To keep every table in workspace.dldg,
# this version uses bronze_ and silver_ table-name prefixes instead, so no new schemas are needed.
# With Unity Catalog you'd normally use <catalog>.<schema>.<table>, e.g.:
# spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
# spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

*[Cell 44]*

## COPY (CLONE) Tables between Databases (Schemas)
> We will be copying `workspace.dldg.covid_nyt_by_date` using DEEP CLONE into `workspace.dldg.bronze_covid_nyt_by_date` and `workspace.dldg.silver_covid_nyt_by_date` (prefixes stand in for separate bronze/silver schemas)
> This functionality is available in the Databricks runtime as [CLONE](https://docs.databricks.com/delta/clone.html). [Shallow Cloning](https://docs.delta.io/latest/delta-utility.html#shallow-clone-a-delta-table) is available at the time of writing.

In [ ]:
# [Cell 45]
from delta.tables import DeltaTable

# slim version of https://github.com/MrPowers/mack/blob/main/mack/__init__.py#L288
def copy_table(delta_table: DeltaTable, target_table: str):
    details = (
        delta_table
        .detail()
        .select("partitionColumns", "properties")
        .collect()[0]
    )
    # Databricks adds properties it manages itself (row tracking columns, parquet format); they can't be
    # passed back in as write options (DELTA_UNKNOWN_CONFIGURATION), so copy everything else
    properties = {k: v for k, v in details["properties"].items()
                  if not k.startswith(("delta.rowTracking.", "delta.parquet."))}
    (
        table_to_copy.toDF().write.format("delta")
        .partitionBy(details["partitionColumns"])
        .options(**properties)
        .saveAsTable(target_table)
    )


In [ ]:
# [Cell 46]
# copy the default table and write into both bronze and silver
table_to_copy = DeltaTable.forName(spark, "workspace.dldg.covid_nyt_by_date")
bronze_table = "workspace.dldg.bronze_covid_nyt_by_date"
silver_table = "workspace.dldg.silver_covid_nyt_by_date"

copy_table(table_to_copy, bronze_table)
copy_table(table_to_copy, silver_table)

*[Cell 47]*

## Using Shallow Clone to Create a Metadata-Only Copy of a Table
Reference Link: https://docs.delta.io/latest/delta-utility.html#shallow-clone-a-delta-table

The next example is extra content outside of the book materials for chapter 6. We'll discover how to shallow clone a table using both the path on disk, as well as from table to table references (for managed tables).

In [ ]:
# [Cell 48]
# Not supported here: with Unity Catalog, shallow clone source and target must both be managed
# tables (CANNOT_SHALLOW_CLONE_NON_UC_MANAGED_TABLE_AS_SOURCE_OR_TARGET), so cloning to a path fails.
# The next cell clones into a managed table, which works. Original Docker version for reference:
# # Use Shallow Clone to Create a Metadata Only Copy of the Table using the table location on disk
# src_location = DeltaTable.forName(spark, "workspace.dldg.covid_nyt_by_date").detail().first()["location"]
# dest_location = DeltaTable.forName(spark, "workspace.dldg.silver_covid_nyt_by_date").detail().first()["location"]
# #print(f"source_table:{src_location}\ndestination_table_location:{dest_location}")
#
# src_location_fmt = str(src_location).replace("file:", "")
# # steal the silver.db location from the copy table, and just add _clone to the tablename
# dest_location_clone = str(dest_location).replace("file:","")+'_clone'
# #print(f"src:{src_location_fmt}, dest:{dest_location_clone}")
#
#
# spark.sql(f"CREATE TABLE IF NOT EXISTS delta.`{dest_location_clone}` SHALLOW CLONE delta.`{dest_location_clone}`")

In [ ]:
# [Cell 49]
spark.catalog.setCurrentDatabase("dldg")
spark.catalog.listTables()
spark.sql("show tables").show()

# On the first pass, without writing to the managed table location, you won't be able to see the new cloned table in the table
# list. This is one way to work with cloned data where you are not "broadcasting" the table into the managed table space. When you are ready
# you can always create a managed table using the location.

In [ ]:
# [Cell 50]
# It is worth noting that you can CREATE a managed table over an existing non-managed table.
# Observe the WARNING when running the next statement.
spark.sql("CREATE TABLE IF NOT EXISTS workspace.dldg.silver_covid_nyt_by_date_clone SHALLOW CLONE workspace.dldg.covid_nyt_by_date")

# if you try to replace a CLONED table, you will get an exception 
# (DeltaIllegalStateException): The clone destination table is non-empty: Please TRUNCATE or DELETE before running CLONE...
# this is to protect the integrity of the clone, the expectation for a SHALLOW CLONE is that it provides metadata only changes
# as the source table is still the reference for the data.

# To see the behavior in action, try
# spark.sql("CREATE OR REPLACE TABLE workspace.dldg.silver_covid_nyt_by_date_clone SHALLOW CLONE workspace.dldg.covid_nyt_by_date")

In [ ]:
# [Cell 51]
# after replacing the table clone, you'll see the table in the local table list
spark.catalog.setCurrentDatabase("dldg")
spark.catalog.listTables()

*[Cell 52]*

## Removing Partitions using Conditional Delete at the Partition Boundary

In [ ]:
# [Cell 53]
## Remove a partition from the silver table so we can repair the table with our bronze table
silver_dt = DeltaTable.forName(spark, "workspace.dldg.silver_covid_nyt_by_date")
silver_dt.delete(col("date") == "2021-02-17")

# Note: (if you delete, and then immediately vacuum, you will not be able to restore your table)
# vacuum to remove the physical data from the table
#spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled","false")
#silver_dt.vacuum(retentionHours=0)
#spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled","true")

*[Cell 54]*

## Using ReplaceWhere to do Conditional Repairs

In [6]:
# [Cell 55]
recovery_table = spark.table("workspace.dldg.bronze_covid_nyt_by_date")
partition_col = "date"
table_to_fix = "workspace.dldg.silver_covid_nyt_by_date"

(recovery_table.where(col("date") == "2021-02-17").write.format("delta")
 .mode("overwrite")
 .option("replaceWhere", f"{partition_col} == '2021-02-17'")
 .saveAsTable("workspace.dldg.silver_covid_nyt_by_date")
)

23/06/16 21:10:20 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
23/06/16 21:10:20 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
23/06/16 21:10:20 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
23/06/16 21:10:20 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


*[Cell 56]*

## Restoring Tables to a Prior Version



In [ ]:
# [Cell 57]
dt = DeltaTable.forName(spark, "workspace.dldg.silver_covid_nyt_by_date")
dt.history(10).select("version", "timestamp", "operation").show()
dt.restoreToVersion(0)

*[Cell 58]*

## Cleaning up our Delta Tables using Vacuum

In [ ]:
# [Cell 59]
# Serverless doesn't allow disabling spark.databricks.delta.retentionDurationCheck.enabled, so a
# 0-hour retention is rejected (DELTA_VACUUM_RETENTION_PERIOD_TOO_SHORT). Using the 7-day default instead.
DeltaTable.forName(spark, "workspace.dldg.nonoptimal_covid_nyt").vacuum()

In [ ]:
# [Cell 60]
spark.sql("select distinct(date) as date from workspace.dldg.covid_nyt_by_date order by date desc").show(200)

In [ ]:
# [Cell 61]
spark.sql("select count(distinct(date)) from workspace.dldg.covid_nyt_by_date").show()